In [1]:
!pip install transformers accelerate mlflow torch scikit-learn evaluate 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 66.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 

In [2]:

from transformers import AutoTokenizer ,AutoModelForSequenceClassification
import mlflow
from transformers.integrations import MLflowCallback
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import load_dataset, DatasetDict 
from sklearn.model_selection import train_test_split
import pandas as pd


2025-11-20 13:12:48.705118: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763644368.899149      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763644368.951773      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
MODEL_NAME = "Shushant/nepaliBERT"
NUM_LABELS = 3 # 0 and 1 ,2 sentiment


In [4]:
tokenizer = AutoTokenizer.from_pretrained("AlgoAlchemist/sentimentclassifiernepali")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/589 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at Shushant/nepaliBERT and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
df = pd.read_csv("/kaggle/input/sentimentanalysisnepalidataset/final_preprocessed_012.csv")

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [6]:
df.drop(["cleaned_tweet"], axis =1 , inplace = True)

In [7]:
df['preprocessed'] = df['preprocessed'].fillna(1)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102141 entries, 0 to 102140
Data columns (total 2 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   sentiment     102141 non-null  int64 
 1   preprocessed  102141 non-null  object
dtypes: int64(1), object(1)
memory usage: 1.6+ MB


In [9]:
# Check the frequency of each sentiment label
print(df['sentiment'].value_counts())


sentiment
0    34047
1    34047
2    34047
Name: count, dtype: int64


In [10]:

import os 
train_df, val_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42, 
    stratify=df["sentiment"]
)

train_file = "temp_train.csv"
val_file = "temp_val.csv"
train_df.to_csv(train_file, index=False)
val_df.to_csv(val_file, index=False)

# 2. Load files into a HF DatasetDict object
raw_datasets = load_dataset("csv", data_files={"train": train_file, "validation": val_file})


def tokenize_fn(batch):
    
    return tokenizer(batch["preprocessed"], truncation=True, padding="max_length", max_length=128)

tokenized = raw_datasets.map(tokenize_fn, batched=True)


columns_to_keep = ["input_ids", "attention_mask", "sentiment"]
tokenized = tokenized.remove_columns([c for c in tokenized["train"].column_names if c not in columns_to_keep])


tokenized = tokenized.rename_column("sentiment", "label") 
tokenized.set_format(type="torch")


train_dataset = tokenized["train"]
eval_dataset = tokenized["validation"]

# --- Cleanup ---
os.remove(train_file)
os.remove(val_file)



Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/81712 [00:00<?, ? examples/s]

Map:   0%|          | 0/20429 [00:00<?, ? examples/s]

In [11]:
from transformers import TrainingArguments, Trainer

In [12]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    # Ensure average="binary" is appropriate for your 0/1 sentiments
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro") 
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

In [13]:
# ...existing code...
training_args = TrainingArguments(
    output_dir="./outputs",
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    fp16=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to=["mlflow"],   # enable reporting to MLflow
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[MLflowCallback()]
)

# Optionally create an explicit MLflow run to attach extra tags/params
with mlflow.start_run() as run:
    mlflow.log_param("model_name", "AlgoAlchemist/sentimentclassifiernepali")
    mlflow.log_param("num_labels", 3)
    trainer.train()
    metrics = trainer.evaluate()
    mlflow.log_metrics(metrics)
    trainer.save_model("kaggle/working/outputs/best_model")
# ...existing code...

/tmp/ipykernel_48/1334890839.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
You are adding a <class 'transformers.integrations.integration_utils.MLflowCallback'> to the callbacks of this Trainer, but there is already one. The currentlist of callbacks is
:DefaultFlowCallback
MLflowCallback
/usr/local/lib/python3.11/dist-packages/mlflow/tracking/_tracking_service/utils.py:140: FutureWarning: Filesystem tracking backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri, store_uri)
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.628400,0.631022,0.733320,0.738219,0.733321,0.730772
2,0.531800,0.593362,0.754369,0.755161,0.754371,0.754222
3,0.458200,0.614436,0.757697,0.758425,0.757700,0.756641
4,0.393400,0.646196,0.757697,0.757449,0.757699,0.757012


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


In [4]:
!zip -r all_working_files.zip .

  adding: mlruns/ (stored 0%)
  adding: mlruns/.trash/ (stored 0%)
  adding: mlruns/0/ (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/ (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/params/ (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/params/save_only_model (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/params/deepspeed (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/params/include_num_input_tokens_seen (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/params/finetuning_task (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/params/liger_kernel_config (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/params/add_cross_attention (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/params/fp16_full_eval (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/params/torchdynamo (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/param

In [5]:
!ls model_outputs.zip


ls: cannot access 'model_outputs.zip': No such file or directory


In [6]:
from IPython.display import FileLink

# Generates a link to download the file created above
FileLink(r'all_working_files.zip')

/kaggle/working/all_working_files.zip

In [30]:
!cd working

/bin/bash: line 1: cd: working: No such file or directory


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


rm: cannot remove 'kaggle': No such file or directory
rm: cannot remove 'mlruns': No such file or directory
rm: cannot remove 'outputs': No such file or directory


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [1]:
!pip install 'optimum[onnxruntime]' 


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 81.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 67.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 67.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.3/162.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install transformers

In [2]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer


2025-11-20 14:33:42.665848: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763649222.926403      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763649222.993882      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Multiple distributions found for package optimum. Picked distribution: optimum-onnx


In [3]:
model_id = "/kaggle/working/kaggle/working/outputs/best_model"
save_directory = "/kaggle/working/onnx_model"

In [4]:
ort_model = ORTModelForSequenceClassification.from_pretrained(model_id, export=True)
tokenizer = AutoTokenizer.from_pretrained(model_id)

ort_model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

/usr/local/lib/python3.11/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


('/kaggle/working/onnx_model/tokenizer_config.json',
 '/kaggle/working/onnx_model/special_tokens_map.json',
 '/kaggle/working/onnx_model/vocab.txt',
 '/kaggle/working/onnx_model/added_tokens.json',
 '/kaggle/working/onnx_model/tokenizer.json')

In [6]:
!pip install hf-cli


In [9]:
!hf auth login --token 


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `cli` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `cli`


In [10]:
!huggingface-cli repo create nepalisentimentbert


⚠️  Warning: 'huggingface-cli repo' is deprecated. Use 'hf repo' instead.
Successfully created AlgoAlchemist/nepalisentimentbert on the Hub.
Your repo is now available at https://huggingface.co/AlgoAlchemist/nepalisentimentbert


In [11]:
!huggingface-cli upload nepalisentimentbert /kaggle/working/onnx_model 


⚠️  Warning: 'huggingface-cli upload' is deprecated. Use 'hf upload' instead.
Start hashing 6 files.
Finished hashing 6 files.
Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...ing/onnx_model/model.onnx:   0%|              |  552kB /  438MB            

Processing Files (0 / 1)      :   0%|              |  552kB /  438MB,  690kB/s  
New Data Upload               :   1%|              |  552kB / 67.1MB,  690kB/s  

Processing Files (0 / 1)      :   2%|▎             | 9.93MB /  438MB, 9.93MB/s  
New Data Upload               :  15%|██            | 9.93MB / 67.1MB, 9.93MB/s  

Processing Files (0 / 1)      :   9%|█▏            | 37.5MB /  438MB, 31.3MB/s  
New Data Upload               :  28%|███▉          | 37.5MB /  134MB, 31.3MB/s  

Processing Files (0 / 1)      :  14%|█▉            | 61.2MB /  438MB, 43.7MB/s  
New Data Upload               :  46%|██████▍       | 61.2M